# 08.05 · GraphRAG & structured data

**Chapter:** Retrieval-Augmented Generation · **Milestone:** M3

## Goals
- [ ] Text-to-SQL: answer questions over a database, safely (read-only, validated, limited)
- [ ] Build a small knowledge graph from text with an LLM and answer multi-hop questions over it
- [ ] Know when graphs/SQL beat vector search

## Resources
- [Microsoft GraphRAG](https://microsoft.github.io/graphrag/) · [LightRAG](https://github.com/HKUDS/LightRAG) — free
- [Neo4j – GraphRAG manifesto & guides](https://neo4j.com/blog/genai/graphrag-manifesto/) — free
- [Spider / BIRD text-to-SQL benchmarks](https://bird-bench.github.io/) — free

Vector search answers "find passages like this". It's bad at **aggregation** ("how many orders last month?") and **multi-hop relations**
("which colleagues of X's manager worked on Y?"). SQL and graphs handle those.

In [ ]:
import sqlite3
from typing import Literal

import networkx as nx
from pydantic import BaseModel, Field

from aieng import DATA, rag
from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}


def parse(messages, schema, **kw):
    kw.setdefault("max_tokens", 1500)  # cap: small models can loop inside JSON output
    return c.chat.completions.parse(model=MODEL, messages=messages, response_format=schema, temperature=0, **EXTRA, **kw).choices[0].message.parsed

## Part A — Text-to-SQL
### A1. A small shop database

In [ ]:
db_path = DATA / "shop.sqlite"
db_path.unlink(missing_ok=True)
con = sqlite3.connect(db_path)
con.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT);
CREATE TABLE products  (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL);
CREATE TABLE orders    (id INTEGER PRIMARY KEY, customer_id INTEGER REFERENCES customers(id),
                        product_id INTEGER REFERENCES products(id), quantity INTEGER, order_date TEXT);
INSERT INTO customers VALUES (1,'Ana','Lima'),(2,'Luis','Cusco'),(3,'Marta','Lima'),(4,'Jorge','Arequipa');
INSERT INTO products VALUES (1,'Coffee beans','food',45.0),(2,'Grinder','equipment',120.5),(3,'Mug','equipment',15.0),(4,'Chocolate','food',8.5);
INSERT INTO orders VALUES (1,1,1,2,'2026-03-01'),(2,1,3,1,'2026-03-03'),(3,2,2,1,'2026-03-10'),(4,3,4,6,'2026-04-02'),
                          (5,3,1,1,'2026-04-15'),(6,4,3,4,'2026-04-20'),(7,2,4,3,'2026-05-05'),(8,1,2,1,'2026-05-09');
""")
con.commit(); con.close()
SCHEMA = "\n".join(r[0] for r in sqlite3.connect(db_path).execute("SELECT sql FROM sqlite_master WHERE type='table'"))
print(SCHEMA)

### A2. Generate SQL — and never trust it
Defence in depth: a **read-only connection** (the DB itself refuses writes), a check that it's a single `SELECT`, and a row limit.
An LLM that writes SQL is an LLM that can be prompt-injected into writing `DROP TABLE` (chapter 11).

In [ ]:
class SQL(BaseModel):
    reasoning: str = Field(description="which tables/joins are needed")
    query: str = Field(description="a single SQLite SELECT statement")


def run_readonly(query, max_rows=50):
    q = query.strip().rstrip(";")
    if not q.lower().startswith(("select", "with")) or ";" in q:
        raise ValueError(f"refusing non-SELECT or multi-statement SQL: {q[:80]}")
    ro = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)  # writes fail at the DB level
    try:
        return ro.execute(q).fetchmany(max_rows)
    finally:
        ro.close()


def text_to_sql(question):
    sql = parse([{"role": "system", "content": f"You write SQLite queries for this schema:\n{SCHEMA}"},
                 {"role": "user", "content": question}], SQL)
    return sql.query, run_readonly(sql.query)


print(run_readonly("SELECT name FROM customers LIMIT 2"))
for bad in ["DROP TABLE orders", "SELECT 1; DELETE FROM orders"]:
    try:
        run_readonly(bad)
    except ValueError as e:
        print("blocked:", e)

### A3. Evaluate with gold answers
Compare **result sets**, not SQL strings (many queries are equivalent).

In [ ]:
CASES = [
    ("How many customers live in Lima?", "SELECT COUNT(*) FROM customers WHERE city='Lima'"),
    ("What is the total revenue from all orders?", "SELECT SUM(o.quantity*p.price) FROM orders o JOIN products p ON p.id=o.product_id"),
    ("Which customer spent the most? Return the name only.", "SELECT c.name FROM orders o JOIN products p ON p.id=o.product_id JOIN customers c ON c.id=o.customer_id GROUP BY c.id ORDER BY SUM(o.quantity*p.price) DESC LIMIT 1"),
    ("How many mugs were sold in total?", "SELECT SUM(quantity) FROM orders WHERE product_id=3"),
    ("List the names of products never ordered by anyone in Cusco.", "SELECT name FROM products WHERE id NOT IN (SELECT product_id FROM orders o JOIN customers c ON c.id=o.customer_id WHERE c.city='Cusco')"),
    ("What was the revenue in April 2026?", "SELECT SUM(o.quantity*p.price) FROM orders o JOIN products p ON p.id=o.product_id WHERE order_date LIKE '2026-04%'"),
]


def normalise(rows):
    return sorted(tuple(round(v, 2) if isinstance(v, float) else v for v in r) for r in rows)


correct = 0
for q, gold_sql in CASES:
    try:
        sql, rows = text_to_sql(q)
        ok = normalise(rows) == normalise(run_readonly(gold_sql))
    except Exception as e:
        sql, rows, ok = f"ERROR {e}", None, False
    correct += ok
    print(f"{'✓' if ok else '✗'} {q}\n    {sql}\n    -> {rows}")
print(f"\n{correct}/{len(CASES)} correct")

Production text-to-SQL adds: table/column descriptions, sample rows, few-shot examples of tricky queries, retries on SQL errors (feed the error
back), and a semantic layer (pre-defined metrics) so the model doesn't re-derive "revenue" differently each time.

## Part B — A small knowledge graph
### B1. Extract (subject, relation, object) triples with an LLM

In [ ]:
class Triple(BaseModel):
    subject: str
    relation: str = Field(description="short verb phrase, e.g. 'born in', 'married', 'son of'")
    object: str


class Triples(BaseModel):
    triples: list[Triple]


passages = [t for _, _, t in rag.search("Abraham Lincoln family wife children parents", k=8)]
G = nx.MultiDiGraph()
for p in passages:
    try:
        triples = parse([{"role": "user", "content": f"Extract factual (subject, relation, object) triples about people, places and dates. "
                                                 f"Use full names.\n\n{p}"}], Triples).triples
    except Exception as e:  # truncated / invalid JSON: skip this passage, keep going
        print("skipped a passage:", type(e).__name__)
        continue
    for t in triples:
        G.add_edge(t.subject.strip(), t.object.strip(), relation=t.relation.strip())
print(G.number_of_nodes(), "entities |", G.number_of_edges(), "relations")
for u, v, d in list(G.edges(data=True))[:12]:
    print(f"  {u} --{d['relation']}--> {v}")

### B2. Answer by graph neighbourhood
Find entities mentioned in the question, collect facts within 2 hops, give those facts to the LLM. Multi-hop answers emerge from paths the
vector search might never retrieve together.

In [ ]:
def graph_context(question, hops=2, max_facts=40):
    seeds = [n for n in G.nodes if any(part.lower() in question.lower() for part in n.split() if len(part) > 3)]
    sub = set(seeds)
    for _ in range(hops):
        sub |= {nb for n in list(sub) for nb in nx.all_neighbors(G, n)}
    facts = [f"{u} {d['relation']} {v}" for u, v, d in G.subgraph(sub).edges(data=True)]
    return seeds, facts[:max_facts]


q = "Who were the parents of Abraham Lincoln's wife?"
seeds, facts = graph_context(q)
print("seeds:", seeds[:5])
print("\n".join(facts[:15]))
print("\nA:", c.chat.completions.create(model=MODEL, temperature=0, **EXTRA, messages=[{"role": "user", "content":
      "Answer using only these facts. If they're insufficient, say so.\n" + "\n".join(facts) + f"\n\nQuestion: {q}"}]).choices[0].message.content)

**Check the answer.** Small models often conflate entities here (e.g. giving Lincoln's *own* parents as his wife's).
Expect a messy graph: duplicate entities ("Lincoln" vs "Abraham Lincoln"), inconsistent relation names, missed facts. **Entity resolution**
and a fixed relation schema are most of the real work. Microsoft GraphRAG adds community detection + LLM summaries per community to answer
global questions ("what are the main themes?"); LightRAG is a lighter alternative; Neo4j / Kùzu / FalkorDB store graphs at scale.

## When to use what
| Question type | Best tool |
|---|---|
| "What does the doc say about X?" | vector / hybrid RAG |
| counts, sums, filters, rankings over records | **text-to-SQL** (or pre-built API/tools) |
| relationships, multi-hop, "connect the dots" | **knowledge graph** |
| "summarise themes across the whole corpus" | GraphRAG communities / map-reduce summarisation |

## Exercises
1. Add 6 harder SQL questions (dates, NULL handling, ties). Feed SQL errors back to the model for one retry. How much does it help?
2. Do entity resolution: merge nodes whose names are substrings of others ("Lincoln" → "Abraham Lincoln"). Does the answer improve?
3. `uv add lightrag-hku` (or Microsoft `graphrag`) and run it over 50 passages. Compare with your hand-built graph.

In [ ]:
# your code here